# MRAi_P3 — Fase 2 en Colab (GPU NVIDIA)
Fine-tune 2D-US elongado. Guía completa en `COLAB.md`. Ejecuta celdas en orden 0→6.

## 0 — GPU check
Si `GPUs: []`, cambia el runtime a GPU antes de seguir (Entorno → Tipo de ejecución).

In [ ]:
!ls /content/drive/MyDrive/mrai_p3/colab 2>/dev/null || echo 'monta Drive en celda 1 primero'
import sys; sys.path.insert(0, '/content/drive/MyDrive/mrai_p3/colab/tools')
exec(open('/content/drive/MyDrive/mrai_p3/colab/tools/verify_gpu.py').read())

## 1 — Setup: pip + Drive + rutas
Instala deps, monta Drive y define BASE/DATA/OUT en Drive (sobrevive al reciclaje de la VM).

In [ ]:
%pip install -q -r /content/drive/MyDrive/mrai_p3/colab/requirements-colab.txt
from google.colab import drive; drive.mount('/content/drive')
import os
BASE='/content/drive/MyDrive/mrai_p3'; COLAB=f'{BASE}/colab'; DATA='/content/dataset'; STAGED='/content/staged'; OUT=f'{BASE}/out_elong'
os.makedirs(OUT, exist_ok=True)
print(BASE, COLAB, DATA, STAGED, OUT)

## 2 — Stage data
Descomprime `dataset.zip` (spectra+labels+dataset.json+manifest.jsonl, ~2.5 GB) y `DNN_0.zip` (--init).

In [ ]:
!unzip -q -o /content/drive/MyDrive/mrai_p3/dataset.zip -d /content/dataset_tmp && mkdir -p /content/dataset && cp -r /content/dataset_tmp/*/* /content/dataset/ 2>/dev/null || cp -r /content/dataset_tmp/* /content/dataset/
!unzip -q -o /content/drive/MyDrive/mrai_p3/DNN_0.zip -d /content/init
!ls /content/dataset/ && ls /content/dataset/spectra | head -3 && ls /content/init/

## 3 — Ingesta Fase 1 (ruta jres)
Regenera `manifest.json` con `source=/content/dataset`. Primero `--validate-only` + probe, luego ingesta real.

In [ ]:
%cd /content/drive/MyDrive/mrai_p3/colab
!python scripts/jres_ingest.py --dataset /content/dataset --out /content/staged --validate-only
!python scripts/jres_ingest.py --dataset /content/dataset --out /content/staged
!cat /content/staged/manifest.json | head -c 600

## 4 — Smoke test
Valida modelo + I/O con 1 época pequeña antes de gastar horas.

In [ ]:
!python scripts/train_model.py --data /content/staged --max-pts 4096 --epochs 1 --batch 4096 --out /content/smoke

## 4b — Pre-extraccion (acelerador grande, ~6 min una vez)
Extrae `(X,y)` fijos por espectro con semilla fija. Quita el `split_2d` (~1 s/espectro) de CADA epoca: pasa de ~15 min/epoca a ~1 s de I/O.
Coste: ~6 min y ~1.5 GB. M real ~2900 puntos/espectro (580 positivos x5), no 65536.
Nota: fija el muestreo de fondo entre epocas; compensa con el shuffle por epoca (ya activo).

In [ ]:
!python scripts/preextract.py --dataset /content/dataset --ids /content/staged/manifest.json --out /content/pre --seed 0 --fast
!echo '--- log (ultimas) ---' && tail -n 3 /content/pre/preextract.log
!du -sh /content/pre && ls /content/pre/*.npz | wc -l

## 5 — Fine-tune (recomendado)
Usa `/content/pre` pre-extraido (celda 4b): ~5-10 s/epoca. Con 35 steps/epoca (train) y early-stopping, 30-50 epocas ≈ 10-15 min.
Ruta on-demand (sin pre-extraer, replica literal): cambia `--data /content/pre --preextracted` por `--data /content/staged`; cuesta ~15 min/epoca.
Salida en Drive + `ckpt.keras` para reanudar si Colab corta.

In [ ]:
!python scripts/train_model.py --data /content/pre --preextracted --init /content/init/DNN_0 --lr 1e-4 --batch 65536 --epochs 210 --patience 20 --out /content/drive/MyDrive/mrai_p3/out_elong

## 5b — Reanudar tras corte (solo si Colab se desconectó)
`ckpt.keras` + `training.log` ya están en Drive (celda 5 escribe ahí). Esta celda continúa desde el mejor checkpoint.
OJO: `CSVLogger` sobrescribe `training.log`: la celda lo renombra antes a `training_part1.log` para no perder la curva.

In [ ]:
!ls -lh /content/drive/MyDrive/mrai_p3/out_elong/ckpt.keras && tail -n 5 /content/drive/MyDrive/mrai_p3/out_elong/training.log
!mv /content/drive/MyDrive/mrai_p3/out_elong/training.log /content/drive/MyDrive/mrai_p3/out_elong/training_part1.log 2>/dev/null; echo log-rotated
!python scripts/train_model.py --data /content/staged --resume /content/drive/MyDrive/mrai_p3/out_elong/ckpt.keras --lr 1e-4 --batch 65536 --epochs 210 --patience 20 --out /content/drive/MyDrive/mrai_p3/out_elong

## 6 — Export
Empaqueta el resultado para bajar al Mac y usar con `StoP3` sin cambios.

In [ ]:
!cd /content/drive/MyDrive/mrai_p3 && rm -f out_elong.zip && zip -qr out_elong.zip out_elong && ls -lh out_elong.zip
!ls /content/drive/MyDrive/mrai_p3/out_elong/